# 08 · Shortest paths: Dijkstra, the LP, and its dual

**Goal:** connect the algorithm you know (Dijkstra) to the linear program and its dual. The dual, with **node potentials**, is the key to network interdiction in notebook 12.

### Primal (send one unit of flow from $s$ to $t$)
$$\min\ \sum_{(i,j)}c_{ij}x_{ij}\quad\text{s.t.}\quad \sum_{j}x_{ij}-\sum_{j}x_{ji}=\begin{cases}1&i=s\\-1&i=t\\0&\text{else}\end{cases},\qquad x\ge0 .$$

In matrix form $Nx=e_s-e_t$, where $N$ is the **node–arc incidence matrix**. $N$ is *totally unimodular*, so the LP has an integral optimal vertex: a path.

### Dual
$$\max\ \pi_t-\pi_s\quad\text{s.t.}\quad \pi_j-\pi_i\le c_{ij}\ \ \forall (i,j).$$

The distances $\pi_v=\mathrm{dist}(s,v)$ are dual feasible (triangle inequality) and optimal.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from scipy.optimize import linprog

ARCS = [(0, 1, 4), (0, 2, 2), (2, 1, 1), (1, 3, 5), (2, 4, 7), (1, 4, 6),
        (4, 3, 1), (3, 5, 3), (3, 6, 6), (4, 6, 3), (5, 7, 5), (6, 7, 4)]
n, s, t = 8, 0, 7
m = len(ARCS)
cost = np.array([a[2] for a in ARCS], float)

## 1. Dijkstra from scratch, checked against networkx

In [ ]:
import heapq

def dijkstra(n, arcs, s):
    # arcs: list of (i, j, length) with length >= 0. Returns distances and predecessor array.
    adj = [[] for _ in range(n)]
    for i, j, c in arcs:
        adj[i].append((j, c))
    dist = np.full(n, np.inf); dist[s] = 0.0
    pred = [None] * n
    done = [False] * n
    pq = [(0.0, s)]
    while pq:
        d, u = heapq.heappop(pq)
        if done[u]:
            continue
        done[u] = True                       # u's label is now permanent
        for v, c in adj[u]:
            if d + c < dist[v]:
                dist[v] = d + c; pred[v] = u
                heapq.heappush(pq, (dist[v], v))
    return dist, pred

def path_to(pred, t):
    path = [t]
    while pred[path[-1]] is not None:
        path.append(pred[path[-1]])
    return path[::-1]

In [ ]:
dist, pred = dijkstra(n, ARCS, s)
sp = path_to(pred, t)
print("distances:", dist)
print("shortest path:", sp, " length", dist[t])

G = nx.DiGraph()
G.add_weighted_edges_from(ARCS)
nx_dist = nx.single_source_dijkstra_path_length(G, s)
assert all(np.isclose(dist[v], nx_dist[v]) for v in nx_dist)

## 2. The same problem as an LP

In [ ]:
Ninc = np.zeros((n, m))
for k, (i, j, _) in enumerate(ARCS):
    Ninc[i, k] = 1.0      # arc leaves i
    Ninc[j, k] = -1.0     # arc enters j
rhs = np.zeros(n); rhs[s] = 1.0; rhs[t] = -1.0

res = linprog(cost, A_eq=Ninc, b_eq=rhs, bounds=(0, None), method="highs")
x = res.x
print("LP optimum:", res.fun, "  arcs used:", [ARCS[k][:2] for k in range(m) if x[k] > 0.5])
assert np.isclose(res.fun, dist[t])
assert np.allclose(x, np.round(x))          # integral, as total unimodularity promises

## 3. The dual and complementary slackness

Solve the dual LP explicitly. Then check **complementary slackness**: any arc carrying flow ($x_{ij}>0$) must be *tight*, $\pi_j-\pi_i=c_{ij}$.

In [ ]:
A_ub = np.zeros((m, n))
for k, (i, j, _) in enumerate(ARCS):
    A_ub[k, j] = 1.0; A_ub[k, i] = -1.0           # pi_j - pi_i <= c_ij
obj = np.zeros(n); obj[t] = -1.0                  # maximize pi_t  (pi_s fixed to 0)
bounds = [(0, 0) if v == s else (None, None) for v in range(n)]
dual = linprog(obj, A_ub=A_ub, b_ub=cost, bounds=bounds, method="highs")
pi = dual.x
print("dual optimum:", -dual.fun, "   potentials:", pi.round(3))
assert np.isclose(-dual.fun, dist[t])             # strong duality

for k, (i, j, cij) in enumerate(ARCS):
    if x[k] > 0.5:
        assert np.isclose(pi[j] - pi[i], cij)     # used arcs are tight
print("complementary slackness holds on the path arcs")

# Dijkstra's distances are themselves an optimal dual solution
assert np.all(A_ub @ dist <= cost + 1e-9)

In [ ]:
pos = {0: (0, 1), 1: (1, 2), 2: (1, 0), 3: (2, 2), 4: (2, 0), 5: (3, 2), 6: (3, 0), 7: (4, 1)}
on_path = set(zip(sp[:-1], sp[1:]))
colors = ["C3" if (i, j) in on_path else "0.6" for i, j, _ in ARCS]
plt.figure(figsize=(7, 3.5))
nx.draw_networkx(G, pos, edge_color=colors, width=2, node_color="w", edgecolors="k")
nx.draw_networkx_edge_labels(G, pos, edge_labels={(i, j): c for i, j, c in ARCS})
plt.axis("off"); plt.title("shortest path in red"); plt.show()

## Exercises

1. Implement **Bellman–Ford** and add an arc with negative cost (no negative cycle). Why does Dijkstra fail there but the LP still works?
2. Read off the dual from the primal solver instead: compare `res.eqlin.marginals` to `dist` (mind the sign convention).
3. Time your Dijkstra on a random graph with $10^5$ arcs (`nx.gnm_random_graph`) against networkx.